# 04_데이터준비

# Preparing and Cleaning Data

실제 Data는 Machine Learning에 바로 사용할 수 있는 형태로 주어지지 않는 경우가 많다.

Data에는 다음과 같은 문제가 있을 수 있다.

- Missing Value
- Duplicate Data
- Invalid Value
- Inconsistent Category
- Outlier

이러한 문제를 확인하고 분석 가능한 형태로 만드는 과정을 Data Cleaning이라고 한다.

이번 강의에서는 `customer_data.csv`를 이용하여 실제 Data Cleaning 과정을 수행한다.

전체 과정은 다음과 같다.

> Load → Understand → Find Problems → Clean → Verify

In [1]:
from google.colab import drive

# Google Drive를 Colab에 Mount
drive.mount('/content/drive')

# 강의용 Data가 저장된 Folder

data_path = "/content/drive/My Drive/Colab Notebooks/0_ML/Input/"

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

# CSV File 읽기
df = pd.read_csv(data_path+"customer_data.csv")

# Data 확인
display(df.head())

# Data 크기 확인
print("Shape:", df.shape)

,CustomerID,Age,Gender,City,Income,Purchase
0,1001,56.0,Female,Incheon,32484.0,630.0
1,1002,69.0,Female,Seoul,60918.0,571.0
2,1003,46.0,Male,Seoul,50384.0,446.0
3,1004,32.0,Female,Daegu,44055.0,635.0
4,1005,60.0,Female,Incheon,18399.0,369.0


Shape: (203, 6)


## Understanding the Data

Data Cleaning을 시작하기 전에 먼저 Data의 구조와 내용을 이해해야 한다.

`info()`를 이용하면 다음과 같은 정보를 확인할 수 있다.

- Row의 수
- Column의 수
- Data Type
- Non-Null Data의 수

`describe()`를 이용하면 Numerical Data의 기본적인 Statistical Information을 확인할 수 있다.

- count
- mean
- standard deviation
- minimum
- maximum
- quartile

Data Cleaning의 첫 단계는 Data를 수정하는 것이 아니라 Data를 관찰하는 것이다.

In [4]:
# Data의 구조 확인
df.info()

# Numerical Data의 기초통계 확인
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  203 non-null    int64  
 1   Age         199 non-null    float64
 2   Gender      203 non-null    object 
 3   City        203 non-null    object 
 4   Income      199 non-null    float64
 5   Purchase    203 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 9.6+ KB


,CustomerID,Age,Income,Purchase
count,203.000000,199.000000,199.000000,203.000000
mean,1099.768473,45.130653,46492.814070,531.266010
std,57.812522,18.027281,34749.030516,354.261187
min,1001.000000,-5.000000,4631.000000,82.000000
25%,1050.500000,31.000000,35674.000000,395.000000
50%,1099.000000,44.000000,45285.000000,512.000000
75%,1149.500000,59.000000,54156.500000,615.500000
max,1200.000000,150.000000,500000.000000,5000.000000


## Basic Statistics

Data의 특성을 이해하기 위해 몇 가지 기본적인 Statistics를 사용할 수 있다.

평균 Mean은 다음과 같이 계산한다.

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
$$

Median은 Data를 크기순으로 정렬했을 때 중앙에 위치하는 값이다.

Standard Deviation은 Data가 Mean을 중심으로 얼마나 퍼져 있는지를 나타낸다.

$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}
$$

Mean과 Median의 차이가 매우 크다면 Data에 Extreme Value가 존재하는지 살펴볼 필요가 있다.

In [5]:
# Age의 기초통계
print("Mean:", df["Age"].mean())
print("Median:", df["Age"].median())
print("Standard Deviation:", df["Age"].std())
print("Minimum:", df["Age"].min())
print("Maximum:", df["Age"].max())

# Income의 기초통계
print("\nIncome")
print("Mean:", df["Income"].mean())
print("Median:", df["Income"].median())
print("Minimum:", df["Income"].min())
print("Maximum:", df["Income"].max())

Mean: 45.130653266331656
Median: 44.0
Standard Deviation: 18.027280535634922
Minimum: -5.0
Maximum: 150.0

Income
Mean: 46492.81407035176
Median: 45285.0
Minimum: 4631.0
Maximum: 500000.0


## Missing Values

실제 Dataset에서는 일부 값이 존재하지 않는 경우가 있다.

이를 Missing Value라고 한다.

Pandas에서는 Missing Value가 주로 `NaN`으로 표현된다.

먼저 어떤 Column에 Missing Value가 존재하는지 확인해야 한다.

In [6]:
# 각 Column의 Missing Value 수 확인
print(df.isnull().sum())

# Missing Value가 하나라도 있는 Row 확인
display(df[df.isnull().any(axis=1)])

CustomerID    0
Age           4
Gender        0
City          0
Income        4
Purchase      0
dtype: int64


,CustomerID,Age,Gender,City,Income,Purchase
5,1006,NaN,Male,Incheon,61675.0,466.0
12,1013,41.0,Male,Seoul,NaN,395.0
23,1024,NaN,Male,Daegu,41420.0,660.0
44,1045,20.0,Female,Busan,NaN,364.0
71,1072,NaN,Female,Daegu,34871.0,553.0
98,1099,62.0,Female,Daegu,NaN,637.0
120,1121,NaN,Female,Seoul,28791.0,504.0
150,1151,54.0,Male,Busan,NaN,381.0


## Handling Missing Values

Missing Value를 처리하는 가장 간단한 방법은 해당 Row를 제거하는 것이다.

그러나 Data를 제거하면 사용할 수 있는 Sample의 수도 감소한다.

또 다른 방법은 Missing Value를 다른 값으로 대체하는 것이다.

Numerical Data에서는 Mean이나 Median을 사용할 수 있다.

어떤 방법이 항상 옳은 것은 아니다.

> Missing Value를 어떻게 처리할 것인가는 Data의 의미와 Problem에 따라 결정해야 한다.

In [7]:
# 원본을 보존하고 Cleaning용 DataFrame 생성
clean_df = df.copy()

# Age의 Missing Value를 Median으로 대체
age_median = clean_df["Age"].median()
clean_df["Age"] = clean_df["Age"].fillna(age_median)

# Income의 Missing Value를 Median으로 대체
income_median = clean_df["Income"].median()
clean_df["Income"] = clean_df["Income"].fillna(income_median)

# 처리 결과 확인
print(clean_df.isnull().sum())

CustomerID    0
Age           0
Gender        0
City          0
Income        0
Purchase      0
dtype: int64


## Duplicate Data

같은 Sample이 여러 번 저장되어 있을 수도 있다.

이러한 Duplicate Data는 특정 Sample이 분석 결과에 불필요하게 여러 번 영향을 주게 할 수 있다.

따라서 Duplicate가 존재하는지 확인하고 필요한 경우 제거한다.

In [8]:
# Duplicate Row 수 확인
print("Duplicate Rows:", clean_df.duplicated().sum())

# Duplicate Row 확인
display(clean_df[clean_df.duplicated()])

# Duplicate Row 제거
clean_df = clean_df.drop_duplicates()

print("Shape after removing duplicates:", clean_df.shape)

Duplicate Rows: 3


,CustomerID,Age,Gender,City,Income,Purchase
200,1026,19.0,Female,Gwangju,56808.0,457.0
201,1051,56.0,Male,Busan,24081.0,502.0
202,1076,23.0,Male,Busan,49670.0,571.0


Shape after removing duplicates: (200, 6)


## Invalid Values

Data Type이 Number라고 해서 모든 값이 올바른 것은 아니다.

예를 들어 Age가 다음과 같다고 하자.

25
43
-5
150

모두 Number이므로 Python에서는 정상적인 값이다.

그러나 사람의 Age라는 의미를 고려하면 -5와 150은 의심해야 한다.

따라서 Data Cleaning에서는 Data Type뿐 아니라 값의 의미도 확인해야 한다.

In [11]:
# Age의 범위 확인
print("Minimum Age:", clean_df["Age"].min())
print("Maximum Age:", clean_df["Age"].max())

# 비정상적인 Age 확인
invalid_age = clean_df[
    (clean_df["Age"] < 0) |
    (clean_df["Age"] > 100)
]

display(invalid_age)

# 0~100 범위를 벗어나는 Age 제거
clean_df = clean_df[
    (clean_df["Age"] >= 0) &
    (clean_df["Age"] <= 100)
]

Minimum Age: 18.0
Maximum Age: 74.0


,CustomerID,Age,Gender,City,Income,Purchase


## Inconsistent Categories

같은 의미를 가지는 Category가 서로 다른 방법으로 기록될 수 있다.

예를 들어 `Male`, `male`, `M`은 같은 의미로 사용되었을 수 있다.

그러나 Computer는 이들을 서로 다른 Category로 인식한다.

따라서 Category Data를 사용하기 전에 어떤 값들이 존재하는지 확인하고 표현을 일관되게 만들어야 한다.

In [12]:
# Gender에 존재하는 Category와 개수 확인
print(clean_df["Gender"].value_counts())

# Gender의 Category를 일관되게 변경
clean_df["Gender"] = clean_df["Gender"].replace({
    "male": "Male",
    "M": "Male",
    "female": "Female"
})

# 변경 결과 확인
print(clean_df["Gender"].value_counts())

Gender
Female    105
Male       90
male        1
M           1
female      1
Name: count, dtype: int64
Gender
Female    106
Male       92
Name: count, dtype: int64


## Unnecessary Spaces

String Data에는 불필요한 Space가 포함될 수 있다.

예를 들어 `Seoul`과 ` Seoul `은 사람에게는 같은 City로 보이지만 Computer에서는 서로 다른 String이다.

Pandas의 `str.strip()`을 이용하면 String의 앞뒤에 있는 Space를 제거할 수 있다.

In [13]:
# City에 존재하는 Category와 개수 확인
print(clean_df["City"].value_counts())

# String 앞뒤의 Space 제거
clean_df["City"] = clean_df["City"].str.strip()

# 변경 결과 확인
print(clean_df["City"].value_counts())

City
Busan      45
Gwangju    41
Daegu      40
Seoul      37
Incheon    33
 Seoul      1
Busan       1
Name: count, dtype: int64
City
Busan      46
Gwangju    41
Daegu      40
Seoul      38
Incheon    33
Name: count, dtype: int64


## Outliers

다른 Data와 비교하여 매우 크거나 작은 값을 Outlier라고 한다.

Outlier는 여러 가지 원인으로 발생할 수 있다.

- Data Entry Error
- Measurement Error
- 실제로 발생한 매우 드문 값

따라서 Outlier를 발견했다고 해서 무조건 제거해서는 안 된다.

먼저 그 값이 왜 발생했는지를 판단해야 한다.

In [14]:
# Income이 큰 Sample부터 확인
display(
    clean_df.sort_values("Income", ascending=False).head()
)

# Purchase가 큰 Sample부터 확인
display(
    clean_df.sort_values("Purchase", ascending=False).head()
)

,CustomerID,Age,Gender,City,Income,Purchase
60,1061,37.0,Female,Incheon,500000.0,413.0
31,1032,42.0,Female,Gwangju,70718.0,311.0
163,1164,32.0,Male,Incheon,69386.0,395.0
187,1188,39.0,Male,Incheon,68682.0,773.0
6,1007,38.0,Female,Gwangju,67445.0,384.0


,CustomerID,Age,Gender,City,Income,Purchase
140,1141,41.0,Female,Busan,47437.0,5000.0
117,1118,25.0,Female,Busan,54691.0,946.0
189,1190,19.0,Male,Incheon,32285.0,919.0
115,1116,18.0,Male,Seoul,47703.0,876.0
180,1181,60.0,Male,Incheon,19250.0,838.0


## Finding Outliers with IQR

Outlier를 찾는 방법 중 하나로 Interquartile Range (IQR)를 사용할 수 있다.

$$
IQR = Q_3 - Q_1
$$

일반적으로 다음 범위를 벗어나는 값을 Outlier 후보로 볼 수 있다.

$$
x < Q_1 - 1.5IQR
$$

$$
x > Q_3 + 1.5IQR
$$

이 기준은 Outlier를 자동으로 제거하기 위한 절대적인 규칙이 아니다.

의심스러운 Data를 찾아 추가로 확인하기 위한 기준으로 사용할 수 있다.

In [15]:
# Income의 Q1, Q3, IQR 계산
Q1 = clean_df["Income"].quantile(0.25)
Q3 = clean_df["Income"].quantile(0.75)
IQR = Q3 - Q1

# Outlier 판단 범위 계산
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# 범위를 벗어난 Income 확인
income_outliers = clean_df[
    (clean_df["Income"] < lower) |
    (clean_df["Income"] > upper)
]

print("Lower Bound:", lower)
print("Upper Bound:", upper)
display(income_outliers)

Lower Bound: 9110.125
Upper Bound: 80191.125


,CustomerID,Age,Gender,City,Income,Purchase
29,1030,39.0,Female,Seoul,4631.0,497.0
60,1061,37.0,Female,Incheon,500000.0,413.0


## Verify the Cleaned Data

Data Cleaning이 끝나면 Data를 다시 확인해야 한다.

다음 사항을 확인한다.

- Missing Value가 남아 있는가?
- Duplicate가 남아 있는가?
- Category가 일관되게 표현되어 있는가?
- 값의 범위가 합리적인가?
- Data Cleaning 과정에서 얼마나 많은 Sample이 변경되거나 제거되었는가?

Data Cleaning은 단순히 문제가 있는 Data를 삭제하는 과정이 아니다.

Data의 문제를 발견하고, 적절한 방법으로 처리하고, 그 결과를 다시 확인하는 과정이다.

> Observe → Decide → Clean → Verify

In [16]:
# Cleaning 결과 확인
print("Shape:", clean_df.shape)

# Missing Value 확인
print("\nMissing Values")
print(clean_df.isnull().sum())

# Duplicate 확인
print("\nDuplicate Rows:", clean_df.duplicated().sum())

# Category 확인
print("\nGender")
print(clean_df["Gender"].value_counts())

print("\nCity")
print(clean_df["City"].value_counts())

# Numerical Data의 기초통계 확인
display(clean_df.describe())

Shape: (198, 6)

Missing Values
CustomerID    0
Age           0
Gender        0
City          0
Income        0
Purchase      0
dtype: int64

Duplicate Rows: 0

Gender
Gender
Female    106
Male       92
Name: count, dtype: int64

City
City
Busan      46
Gwangju    41
Daegu      40
Seoul      38
Incheon    33
Name: count, dtype: int64


,CustomerID,Age,Income,Purchase
count,198.000000,198.000000,198.000000,198.000000
mean,1100.949495,45.020202,46502.979798,531.752525
std,57.942712,15.859139,34774.764308,358.503268
min,1001.000000,18.000000,4631.000000,82.000000
25%,1051.250000,32.000000,35765.500000,393.500000
50%,1101.500000,44.000000,45285.000000,512.500000
75%,1150.750000,58.750000,53535.750000,615.750000
max,1200.000000,74.000000,500000.000000,5000.000000


## Save Clean Data

Data Cleaning이 완료되면 Cleaned Data를 새로운 CSV File로 저장할 수 있다.

원본 Data를 직접 변경하기보다는 Raw Data와 Cleaned Data를 별도로 보관하는 것이 좋다.

> Raw Data → Data Cleaning → Clean Data

In [17]:
# Cleaned Data를 새로운 CSV File로 저장
clean_df.to_csv(data_path+"customer_data_clean.csv", index=False)

print("Saved: customer_data_clean.csv")

Saved: customer_data_clean.csv


### HW_04

`Input/customer_data.csv`를 분석하여 다음 질문에 답하시오.

- Missing Value가 있는 Column Name과 각각의 Missing Value 수를 적으시오.
- 완전히 동일한 Duplicate Row는 몇 개인지 적으시오.
- `Age`가 0보다 작거나 100보다 큰 Row의 `CustomerID`를 모두 적으시오.
- `Gender`에 저장되어 있는 서로 다른 값을 모두 적으시오.
- `Income`의 Mean과 Median을 각각 적으시오.

답만 제출하시오.